Visualization routine to read any `*.tsv`output file and plot it over time.
Not used for the thesis, but can be helpful.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import interp1d
from ipywidgets import interact, FloatSlider

In [9]:
def read_solution_file(filename):
    """
    Reads the Trixi-like solution file into a dict:
    {
        timestep: {
            't': float,
            'x': np.ndarray,
            'w': np.ndarray  # shape (n_points, n_vars)
        },
        ...
    }
    """
    data = {}
    timestep = None
    x_vals = []
    u_vals = []

    with open(filename, "r") as f:
        counter = 0
        for line in f:
            line = line.strip()
            if not line:
                continue

            if line.startswith("# timestep"):
                # Save previous timestep if available
                if timestep is not None:
                    data[counter] = {
                        "t": time,
                        "x": np.array(x_vals),
                        "u": np.array(u_vals)
                    }
                    counter += 1
                # Parse new header
                parts = line.replace(",", "").split()
                timestep = int(parts[1].split("=")[-1]) if "timestep=" in line else int(parts[1])
                points = int(parts[2].split("=")[-1])
                time = float(parts[3].split("=")[-1])
                x_vals, u_vals = [], []

            elif not line.startswith("#"):
                vals = list(map(float, line.split()))
                x_vals.append(vals[0])
                u_vals.append(vals[1:])  # u^(0..n)

        # Save last timestep
        if timestep is not None:
            data[counter] = {
                "t": time,
                "x": np.array(x_vals),
                "u": np.array(u_vals)
            }
            counter += 1

    return data

In [10]:
def interpolate_solution(data, time_query, times, timesteps):
    """Linear interpolation in time for all w's"""
    # Clamp outside range
    if time_query <= times.min():
        return data[timesteps[0]]
    if time_query >= times.max():
        return data[timesteps[-1]]

    # indices around query
    idx_right = np.searchsorted(times, time_query)
    idx_left = idx_right - 1
    t0, t1 = times[idx_left], times[idx_right]
    u0, u1 = data[timesteps[idx_left]]["u"], data[timesteps[idx_right]]["u"]
    x0 = data[timesteps[idx_left]]["x"]

    # linear interpolation in time
    alpha = (time_query - t0) / (t1 - t0)
    u_interp = (1 - alpha) * u0 + alpha * u1

    return {"t": time_query, "x": x0, "u": u_interp}

In [15]:
def plot_solution_time(data, time_query, times, timesteps):
    sol = interpolate_solution(data, time_query, times, timesteps)
    x, u, t = sol["x"], sol["u"], sol["t"]

    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True)
    axes[0].plot(x, u[:,0], label="U_(000)", color='blue')
    axes[0].plot(x, u[:,1], label="U_(100)", color='orange')
    axes[0].plot(x, u[:,2], label="U_(200)", color='green')
    axes[0].plot(x, u[:,3], label="U_(020)", color='purple')
    axes[0].set_title(f"t={t:.4f}")
    axes[0].set_xlabel("X")
    axes[0].grid()
    axes[0].legend()

    axes[1].plot(x, u[:,4], label="U_(300)", color='pink')
    axes[1].plot(x, u[:,5], label="U_(120)", color='brown')
    # axes[1].plot(x, u[:,6], label="$u^{(6)}$", color='gray')
    axes[1].set_title("Higher moments")
    axes[1].set_xlabel("X")
    axes[1].grid()
    axes[1].legend()

    plt.tight_layout()
    plt.show()

In [16]:
# === Usage Example ===
filename = "../out/gram_solution_1D3D.tsv"   # <-- replace with your file
data = read_solution_file(filename)

# Build time → timestep mapping
times = np.array([data[ts]["t"] for ts in sorted(data.keys())])
timesteps = np.array(sorted(data.keys()))

# Interactive slider
interact(
    lambda t: plot_solution_time(data, t, times, timesteps),
    t=FloatSlider(min=times.min(), max=times.max(), step=(times[1]-times[0])/10, value=times.min())
)

interactive(children=(FloatSlider(value=0.0, description='t', max=0.0, step=0.0), Output()), _dom_classes=('wi…

<function __main__.<lambda>(t)>